# Multi-Task Data Collection

이 노트북은 라인 트레이싱 데이터와 손가락 분류 데이터를 하나의 UI에서 수집합니다.
**주의:** VSCode 환경 호환성을 위해 이미지 클릭 대신 슬라이더와 버튼을 이용해 라인 좌표를 수집합니다.

In [1]:
import traitlets
import ipywidgets.widgets as widgets
from IPython.display import display
from jetbot import Camera, bgr8_to_jpeg

import os
import glob
from uuid import uuid1
import cv2
import numpy as np

DATASET_DIR = 'dataset_multi'
LINE_DIR = os.path.join(DATASET_DIR, 'line')
FINGER_DIR = os.path.join(DATASET_DIR, 'finger')
CLASSES = ['0', '1', '2', '3', '4', '5']

os.makedirs(LINE_DIR, exist_ok=True)
for c in CLASSES:
    os.makedirs(os.path.join(FINGER_DIR, c), exist_ok=True)
print('Directories created.')

Directories created.


## 카메라 각도 세팅하기
Data를 수집할 때 사용한 이미지와 맞춰서 추론해야 하므로 카메라 각도를 고정합니다.

In [2]:
from SCSCtrl import TTLServo
import time

def init_arm():
    print("로봇 팔을 초기 위치로 이동합니다...")
    
    # 서보모터 ID 1~5, 각도, 방향(1), 속도(150)
    # 3번 모터는 0도에서 기구적으로 걸리거나 가동 범위를 벗어날 수 있어 50으로 변경합니다.
    TTLServo.servoAngleCtrl(1, 0, 1, 150)
    TTLServo.servoAngleCtrl(2, 50, 1, 150)
    TTLServo.servoAngleCtrl(3, 50, 1, 150)  # 3번 모터 수정 (0 -> 50)
    TTLServo.servoAngleCtrl(4, 0, 1, 150)
    TTLServo.servoAngleCtrl(5, 50, 1, 150)

    time.sleep(1)
    print("초기화 완료!")

init_arm()

Succeeded to open the port
Succeeded to change the baudrate
로봇 팔을 초기 위치로 이동합니다...
초기화 완료!


In [3]:
camera = Camera()

# 원본 카메라 이미지 위젯
image_widget = widgets.Image(format='jpeg', width=224, height=224)
traitlets.dlink((camera, 'value'), (image_widget, 'value'), transform=bgr8_to_jpeg)

# 라인 트레이싱용 타겟(미리보기) 위젯
target_widget = widgets.Image(format='jpeg', width=224, height=224)

mode_select = widgets.ToggleButtons(options=['Line Tracing', 'Finger Classification'], description='Mode:')

# Line variables
x_slider = widgets.FloatSlider(min=-1.0, max=1.0, step=0.001, description='x')
y_slider = widgets.FloatSlider(min=-1.0, max=1.0, step=0.001, description='y')
save_line_button = widgets.Button(description='Save Line Data', button_style='primary')

# Finger UI
class_select = widgets.ToggleButtons(options=CLASSES, description='Fingers:')
save_finger_button = widgets.Button(description='Save Finger', button_style='success')
count_label = widgets.HTML(value='')

def update_counts():
    counts = {}
    for c in CLASSES:
        counts[c] = len(glob.glob(os.path.join(FINGER_DIR, c, '*.jpg')))
    line_count = len(glob.glob(os.path.join(LINE_DIR, '*.jpg')))
    
    html = f'<h4>Counts:</h4><b>Line:</b> {line_count}<br>'
    for c in CLASSES:
        html += f'<b>Finger {c}:</b> {counts[c]}<br>'
    count_label.value = html

def save_finger_snapshot(change):
    c = class_select.value
    filepath = os.path.join(FINGER_DIR, c, f"{c}_{str(uuid1())}.jpg")
    with open(filepath, 'wb') as f:
        f.write(image_widget.value)
    update_counts()
save_finger_button.on_click(save_finger_snapshot)

def save_line_snapshot(change):
    x, y = x_slider.value, y_slider.value
    filepath = os.path.join(LINE_DIR, f"xy_{int(x*112+112):03d}_{int(y*112+112):03d}_{str(uuid1())}.jpg")
    with open(filepath, 'wb') as f:
        f.write(image_widget.value)
    update_counts()
save_line_button.on_click(save_line_snapshot)

# 실시간 미리보기 그리기
def display_xy(camera_image):
    image = np.copy(camera_image)
    x = x_slider.value
    y = y_slider.value
    x_px = int(x * 112 + 112)
    y_px = int(y * 112 + 112)
    image = cv2.circle(image, (x_px, y_px), 8, (0, 255, 0), 3)
    image = cv2.circle(image, (112, 224), 8, (0, 0, 255), 3)
    image = cv2.line(image, (x_px, y_px), (112, 224), (255, 0, 0), 3)
    return bgr8_to_jpeg(image)

traitlets.dlink((camera, 'value'), (target_widget, 'value'), transform=display_xy)

# 슬라이더 값이 바뀔 때 target_widget을 강제 업데이트하도록 추가 연결
def on_slider_change(change):
    target_widget.value = display_xy(camera.value)
x_slider.observe(on_slider_change, names='value')
y_slider.observe(on_slider_change, names='value')

update_counts()

finger_box = widgets.VBox([class_select, save_finger_button])
line_box = widgets.VBox([x_slider, y_slider, save_line_button, widgets.HTML("<b>슬라이더로 X, Y를 맞추고 'Save Line Data' 버튼을 누르세요.</b>")])

def on_mode_change(change):
    if change['new'] == 'Line Tracing':
        finger_box.layout.display = 'none'
        line_box.layout.display = 'block'
        image_widget.layout.display = 'none'
        target_widget.layout.display = 'block'
    else:
        finger_box.layout.display = 'block'
        line_box.layout.display = 'none'
        image_widget.layout.display = 'block'
        target_widget.layout.display = 'none'

mode_select.observe(on_mode_change, names='value')
on_mode_change({'new': mode_select.value})

display(widgets.VBox([mode_select, widgets.HBox([image_widget, target_widget]), finger_box, line_box, count_label]))

In [4]:
camera.stop()